# 9.9 · 权重初始化 / Weight Initialization

> **课程定位 / Where this fits**
> 第 9 课，**Part 9 · 深度学习基础**。
> Lesson 9, **Part 9 · Deep Learning Foundations**.
>
> 训练前要给网络的每个权重一个初始值。这看似小事，**却能决定训练能不能开始**：初始化太大，信号在深层网络里**爆炸**；太小，信号**消失**——无论哪种，梯度都会爆炸/消失，网络学不动。**Xavier / He 初始化**就是为了让信号在每一层保持"不胖不瘦"的方差。
> Before training, every weight needs an initial value. This seems minor but **can decide whether training even starts**: too large and signals **explode** through deep layers; too small and they **vanish** — either way gradients explode/vanish and the net can't learn. **Xavier / He init** keep each layer's signal variance "just right".
>
> 💼 **实战/面试视角**："为什么不能全 0 初始化 / Xavier vs He / 为什么要保方差" 是经典考点。
> 💼 **Practical/interview angle:** "why not init to all zeros / Xavier vs He / why preserve variance" — classic questions.

> 📐 **符号约定 / Notation**
> - $n_{in}, n_{out}$ —— 一层的输入、输出维度 / fan-in, fan-out of a layer
> - $\mathrm{Var}$ —— 方差 / variance

> 💡 **面试相关 / Interview-relevant**
> - "为什么不能把权重全初始化为 0/相同值"（出镜率 ★★★★★，对称性）
> - "Xavier 和 He 初始化的区别"（★★★★★）
> - "初始化和梯度消失/爆炸的关系"（★★★★）
> - "为什么 ReLU 用 He"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解为什么不能全 0/相同初始化（对称性问题）。
   Understand why all-zero/identical init fails (symmetry).
2. 看到坏初始化如何引发激活值爆炸/消失。
   See how bad init causes activations to explode/vanish.
3. 掌握 **Xavier(Glorot)** 与 **He(Kaiming)** 的思想与适用场景。
   Master the idea and use cases of Xavier (Glorot) and He (Kaiming).
4. 实测不同初始化对深层网络信号的影响。
   Empirically measure init's effect on deep-net signals.

## 目录 / TOC
1. [为什么不能全 0：对称性 ⭐](#1)
2. [爆炸与消失：坏初始化的后果 ⭐](#2)
3. [Xavier 与 He：保住方差 ⭐](#3)
4. [实测对比 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么不能全 0：对称性 ⭐ / Why Not All Zeros: Symmetry

最自然的想法是把所有权重设为 0（或同一个常数）。**这是致命错误**。
The most natural idea is to set all weights to 0 (or one constant). **This is fatal.**

如果一层里所有神经元的权重都一样，那么它们对同一个输入会**算出完全相同的输出**，反向传播时也会**收到完全相同的梯度**、做**完全相同的更新**。结果是这一层的所有神经元**永远保持相同**——等价于只有一个神经元，整层的表达能力被浪费。这叫**对称性问题(symmetry breaking failure)**。
If all neurons in a layer share the same weights, they produce **identical outputs** for the same input, receive **identical gradients** in backprop, and make **identical updates**. So all neurons in the layer **stay identical forever** — equivalent to having just one neuron; the layer's capacity is wasted. This is the **symmetry problem**.

**解决办法**：用**随机**初始化打破对称，让每个神经元从不同起点出发、学到不同特征。下面验证全 0 初始化下神经元保持相同。
**Fix:** use **random** init to break symmetry so each neuron starts differently and learns different features. Below we verify neurons stay identical under all-zero init.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn
sns.set_theme(style="whitegrid")
torch.manual_seed(0)

# 一个隐藏层 4 个神经元, 权重全初始化为 0 / hidden layer of 4 neurons, all weights = 0
net = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 1))
nn.init.zeros_(net[0].weight); nn.init.zeros_(net[0].bias)   # 故意全 0 / deliberately all zeros
X = torch.randn(8, 3); y = torch.randn(8, 1)
opt = torch.optim.SGD(net.parameters(), lr=0.1); mse = nn.MSELoss()
for _ in range(50):                                          # 训练 50 步 / train 50 steps
    opt.zero_grad(); mse(net(X), y).backward(); opt.step()
W = net[0].weight.detach().numpy()                          # 第一层权重 (4 神经元 × 3 输入)
print("训练后第一层权重的 4 行(4个神经元):")
print(np.round(W, 4))
print("\n4 个神经元的权重完全相同 → 对称性没被打破 → 整层等价于 1 个神经元(浪费)")
print("结论: 绝不能全 0/相同初始化; 必须随机初始化打破对称")


<a id="2"></a>
## 2. 爆炸与消失：坏初始化的后果 ⭐ / Explosion & Vanishing

随机初始化也有讲究：**随机值的大小(方差)很关键**。把一个信号经过很多层，每层都乘上权重矩阵：
Random init also has nuance: **the scale (variance) of the random values matters**. Push a signal through many layers, each multiplying by a weight matrix:
- 如果权重**偏大**，每层都把信号**放大**一点，几十层后**爆炸**成天文数字（→ 梯度爆炸、NaN）。
  If weights are **too large**, each layer **amplifies** the signal a bit; after dozens of layers it **explodes** (→ exploding gradients, NaN).
- 如果权重**偏小**，每层都**缩小**信号，几十层后**消失**成 0（→ 梯度消失、学不动）。
  If weights are **too small**, each layer **shrinks** the signal; after dozens of layers it **vanishes** to 0 (→ vanishing gradients, no learning).

下面把一个随机输入送进一个 **50 层的深网络**，分别用"太大"和"太小"的初始化，看每层激活值的标准差怎么变化。
Below we feed a random input through a **50-layer deep net** with "too large" and "too small" init, watching how each layer's activation std evolves.


In [ ]:
# 为看清"爆炸/消失"本身, 先用纯线性链(无激活)——激活函数会掩盖爆炸(如 tanh 把输出压在±1)
# To see explosion/vanishing clearly, use a pure linear chain (no activation) — activations mask
# explosion (e.g. tanh bounds output to ±1)
def layer_stds(scale, n_layers=50, width=128):
    x = torch.randn(256, width)                            # 一批输入 / a batch of inputs
    stds = []
    for _ in range(n_layers):
        W = torch.randn(width, width) * scale             # 该层权重, 方差由 scale 控制 / weights, scale controls variance
        x = x @ W                                         # 纯线性传播 / pure linear propagation
        stds.append(x.std().item())                       # 记录这层信号的标准差 / record signal std
    return stds

stds_big = layer_stds(scale=0.12)    # 偏大 / too large
stds_small = layer_stds(scale=0.06)  # 偏小 / too small
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(stds_big, label="scale=0.12 (偏大→爆炸)", lw=2)   # 对数纵轴看清量级 / log-y to see magnitude
ax.semilogy(stds_small, label="scale=0.06 (偏小→消失)", lw=2)
ax.set_xlabel("层数 layer"); ax.set_ylabel("信号标准差 std (对数轴)"); ax.legend()
ax.set_title("坏初始化: 信号在深层网络里指数爆炸或消失(注意纵轴是对数)")
plt.tight_layout(); plt.show()
print(f"偏大: 第50层 std = {stds_big[-1]:.2e}  (每层放大→指数爆炸→梯度爆炸/NaN)")
print(f"偏小: 第50层 std = {stds_small[-1]:.2e}  (每层缩小→指数消失→梯度消失学不动)")
print("关键: 每层把 std 乘以约 scale*sqrt(width); >1 则爆炸, <1 则消失")
print("理想: 让这个倍数≈1, 即每层信号方差保持稳定 → 这正是 Xavier/He 的目标")


<a id="3"></a>
## 3. Xavier 与 He：保住方差 ⭐ / Xavier & He: Preserving Variance

既然问题是"信号方差逐层失控"，思路自然是：**选一个权重方差，让输出方差≈输入方差**（既不放大也不缩小）。数学推导后得到两套公式：
Since the problem is "signal variance spirals out of control layer by layer," the idea is: **pick a weight variance so output variance ≈ input variance** (neither amplify nor shrink). The math gives two recipes:

**Xavier / Glorot 初始化**（适合 tanh、sigmoid 等**关于 0 对称**的激活）：权重方差 $= \dfrac{2}{n_{in}+n_{out}}$，同时考虑前向和反向的方差守恒。
**Xavier / Glorot init** (for tanh, sigmoid — activations **symmetric around 0**): weight variance $= \dfrac{2}{n_{in}+n_{out}}$, balancing forward and backward variance.

**He / Kaiming 初始化**（适合 **ReLU** 及其变体）：权重方差 $= \dfrac{2}{n_{in}}$。因为 ReLU 把一半的输入(负的)清零，等于砍掉一半方差，所以分子用 2 来补偿。
**He / Kaiming init** (for **ReLU** and variants): weight variance $= \dfrac{2}{n_{in}}$. ReLU zeros out half the input (the negatives), halving variance, so the factor 2 compensates.

**一句话记忆**（面试）：**tanh/sigmoid → Xavier；ReLU → He**。现代网络几乎都是 ReLU 家族，所以**默认用 He**（PyTorch 的 `nn.Linear`/`Conv` 默认就是 Kaiming 变体）。
**One-liner** (interview): **tanh/sigmoid → Xavier; ReLU → He**. Modern nets are mostly ReLU-family, so **He by default** (PyTorch's `nn.Linear`/`Conv` default to a Kaiming variant).


In [ ]:
# 用合理初始化重做第 2 节的实验: tanh+Xavier, ReLU+He / redo §2 with proper init
def layer_stds_init(kind, n_layers=50, width=128):
    x = torch.randn(256, width)
    stds = []
    for _ in range(n_layers):
        W = torch.empty(width, width)
        if kind == "xavier":
            nn.init.xavier_normal_(W); act = torch.tanh          # Xavier 配 tanh
        else:
            nn.init.kaiming_normal_(W, nonlinearity="relu"); act = torch.relu  # He 配 ReLU
        x = act(x @ W)
        stds.append(x.std().item())
    return stds

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(layer_stds_init("xavier"), label="Xavier + tanh", lw=2)
ax.plot(layer_stds_init("he"), label="He + ReLU", lw=2)
ax.set_xlabel("层数 layer"); ax.set_ylabel("激活值标准差 std"); ax.legend()
ax.set_title("好初始化: 激活值方差在 50 层里大致保持稳定(不爆炸不消失)")
plt.tight_layout(); plt.show()
print("对比第2节: 激活 std 不再爆炸或塌到 0, 而是稳定在一个区间 → 深层网络也能正常传信号")
print("记忆: tanh/sigmoid → Xavier; ReLU → He (现代默认 He)")


<a id="4"></a>
## 4. 实测对比 + 小结 ⭐ / Empirical Comparison & Summary

最后训一个较深的网络，对比**坏初始化 vs He 初始化**对实际收敛的影响。
Finally, train a deeper net comparing **bad init vs He init** on actual convergence.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data/16.0, digits.target, test_size=0.3,
                                          stratify=digits.target, random_state=0)
Xtr = torch.tensor(X_tr, dtype=torch.float32); ytr = torch.tensor(y_tr)
Xte = torch.tensor(X_te, dtype=torch.float32); yte = torch.tensor(y_te)

def make_deep():                                            # 6 层较深的 ReLU 网络 / a deep-ish ReLU net
    return nn.Sequential(nn.Linear(64,128), nn.ReLU(), nn.Linear(128,128), nn.ReLU(),
                         nn.Linear(128,128), nn.ReLU(), nn.Linear(128,10))

def train(init_kind):
    torch.manual_seed(0); net = make_deep()
    for m in net:
        if isinstance(m, nn.Linear):
            if init_kind == "bad":
                nn.init.normal_(m.weight, std=0.01)        # 太小 → 信号消失 / too small
            else:
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")  # He
            nn.init.zeros_(m.bias)
    opt = torch.optim.SGD(net.parameters(), lr=0.1); ce = nn.CrossEntropyLoss(); losses=[]
    for _ in range(100):
        opt.zero_grad(); loss = ce(net(Xtr), ytr); loss.backward(); opt.step(); losses.append(loss.item())
    acc = (net(Xte).argmax(1)==yte).float().mean().item()
    return losses, acc

l_bad, a_bad = train("bad"); l_he, a_he = train("he")
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(l_bad, label=f"坏初始化 std=0.01 (acc={a_bad:.3f})", lw=2)
ax.plot(l_he, label=f"He 初始化 (acc={a_he:.3f})", lw=2)
ax.set_xlabel("epoch"); ax.set_ylabel("训练损失"); ax.legend()
ax.set_title("初始化对深层网络收敛的影响")
plt.tight_layout(); plt.show()
print(f"坏初始化(太小): 信号/梯度消失, 损失几乎不降, acc={a_bad:.3f}")
print(f"He 初始化:      正常收敛, acc={a_he:.3f}")


```
全 0/相同初始化: 对称性不破→所有神经元相同→整层等价1个→必须随机
方差太大→信号逐层爆炸(梯度爆炸/NaN); 太小→信号逐层消失(梯度消失/学不动)
Xavier(Glorot): Var=2/(n_in+n_out), 配 tanh/sigmoid(关于0对称的激活)
He(Kaiming):    Var=2/n_in, 配 ReLU(补偿ReLU砍掉一半方差); 现代默认
目标: 让每层激活/梯度方差大致守恒, 深网络也能传信号
PyTorch nn.Linear/Conv 默认就是 Kaiming 变体
```

### 💡 面试速查 / Interview cheat-sheet
1. **不能全 0/相同**: 对称性不破, 所有神经元学一样的东西。
   Not all-zero/identical: symmetry unbroken, all neurons learn the same.
2. **坏方差**: 太大→爆炸, 太小→消失(都让梯度失控)。
   Bad variance: too large → explode, too small → vanish (both break gradients).
3. **Xavier**: 2/(n_in+n_out), 配 tanh/sigmoid。
   Xavier: 2/(n_in+n_out), for tanh/sigmoid.
4. **He**: 2/n_in, 配 ReLU(补偿砍掉的一半); 现代默认。
   He: 2/n_in, for ReLU (compensates the halved variance); modern default.
5. **核心思想**: 保住每层信号/梯度的方差。
   Core idea: preserve per-layer signal/gradient variance.

### 下一节 / Next
**9.10 正则化**——深网络容易过拟合。L2 weight decay、Dropout、BatchNorm、早停等正则化手段让模型泛化得更好。
**9.10 Regularization** — deep nets overfit easily. L2 weight decay, Dropout, BatchNorm, early stopping help models generalize.
